In [ ]:
from cryoswath.misc import rgi_code_translator
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from cryoswath_paper_workflow.workflow_support import snakemake_fallback_builder, o1o2reg_tuple_list, project_path

snakemake_fallback = snakemake_fallback_builder(globals())

In [ ]:
df = pd.concat(
    [
        pd.read_csv(
            snakemake_fallback(
                "input",
                input_idx,
                str(project_path("resources", f"calval__compare_arcticdem_strips__temporal_correlation__{rgi_code_translator(f'{o1:02d}', out_type='long_code')}.csv")),
            ),
            index_col=["id", "t1", "t2"],
            parse_dates=True,
            date_format="%Y-%m-%d",
        )
        for input_idx, (o1, _) in enumerate(o1o2reg_tuple_list)
    ],
    keys=[str(o1) for o1, _ in o1o2reg_tuple_list],
).rename(columns={"0": "correlation"})
df.rename_axis(["o1", *df.index.names[1:]], inplace=True)
df = df.assign(
    del_t=(df.index.get_level_values("t1") - df.index.get_level_values("t2"))
)

In [ ]:
df

In [ ]:
sns.relplot(data=df, x="del_t", y="correlation", col="o1", col_wrap=3, alpha=0.7)
plt.show()

In [ ]:
df.set_index("del_t").sort_index()[-500:].plot(rot=45)

In [ ]:
df.set_index("del_t").sort_index().resample("D").mean().interpolate("linear").rolling(
    "20D"
).mean()[-400:].plot(rot=45)
plt.gca().vlines(-369 * (60 * 60 * 24 * 1e9), 0.05, 0.9, color="k")

In [ ]:
tmp = df[df.del_t.between("-400D", "-330D")]
tmp["cat"] = pd.cut(
    tmp.del_t,
    pd.timedelta_range("-410D", "-300D", freq="30D"),
    labels=["~11mo", "~12mo", "~13mo"],
)
sns.boxenplot(data=tmp, y="cat", x="correlation")

In [ ]:
plt.figure(figsize=(12 / 2.54, 12 / 2.54 * 6 / 16))
sns.boxenplot(
    data=df[df.del_t.between("-380D", "-360D")],
    x="correlation",
    showfliers=False,
    color="k",
    lw=0.8,
    fill=False,
)
o1_list = [1, 3, 4, 5, 6, 7, 8, 9]
colors = sns.color_palette("tab10", n_colors=len(o1_list))
palette = dict(zip(map(str, o1_list), colors))
data = df[df.del_t.between("-380D", "-360D")]
sns.swarmplot(
    data=data,
    x="correlation",
    hue="o1",
    palette=palette,
)
plt.yticks([])
plt.xlabel("Correlation")
plt.legend(title="Region number", bbox_to_anchor=(1.02, 1.05), loc="upper left")
plt.tight_layout()
plt.savefig(
    snakemake_fallback(
        "output", "boxen_plt", "figures/figA2__one_year_error_correlation.pdf"
    )
)

In [ ]:
with open(
    snakemake_fallback("output", 0, "results/one_year_error_correlation.txt"), "w"
) as f:
    print(df[df.del_t.between("-380D", "-360D")].correlation.median(), file=f)

*Just FYI, not used in the associated article:*

In [ ]:
tmp = df
tmp["_cat"] = pd.cut(
    tmp.del_t, pd.timedelta_range("-320D", df.del_t.min(), freq="-91D")[::-1]
)
mask = np.logical_or.reduce(
    [
        pd.IntervalIndex(tmp._cat).contains(td)
        for td in pd.timedelta_range("-365D", df.del_t.min(), freq="-365D")[::-1]
    ]
)

tmp2 = tmp[mask].sort_values("_cat", ascending=False)
tmp2["_cat"] = tmp2._cat.apply(
    lambda c: f"~{np.abs(c.right // pd.Timedelta('365D'))}yr"
)
sns.boxenplot(data=tmp2, x="_cat", y="correlation", showfliers=False)